# UN M7 v14 replay plots

This notebook reruns the selected M7 v14 candidate without Optuna. It loads the standalone v14 solver, reads the already-selected parameters from the final CSV, and regenerates diagnostic plots.

## Reduced M7 balance used here

The solver tracks intragranular matrix gas `c`, gas in bulk bubbles `m_b`, gas in dislocation bubbles `m_d`, and the two bubble number densities `N_b`, `N_d`.

For one time step, the gas exchange part is represented by the reduced balances

$$
\frac{dc}{dt}=S_c - g_b c - g_d c + b_b^{gas} m_b + b_d^{gas} m_d + D_g \nabla^2 c,
$$

$$
\frac{dm_b}{dt}=S_b + g_b c - b_b^{gas} m_b,
$$

$$
\frac{dm_d}{dt}=S_d + g_d c - b_d^{gas} m_d.
$$

The capture-only v14 family uses

$$
S_c = \beta, \qquad S_b = 0, \qquad S_d = 0,
$$

so the bulk nucleation mass-coupling term is disabled. In full M7 mode the code can use

$$
S_c = \beta - 2\nu_b, \qquad S_b = 2\nu_b.
$$

The optional gas re-solution population factor is

$$
b_i^{gas}=b_i\phi_i,
$$

but for the v14 `capture_only` family `phi` gas re-solution is disabled, so

$$
b_i^{gas}=b_i.
$$

Bulk bubble number density evolves as

$$
\frac{dN_b}{dt}=\nu_b-b_b\phi_b N_b,
$$

and the dislocation bubble density is reduced by dislocation-bubble coalescence when dislocation bubbles grow:

$$
N_d^{n+1}=\frac{N_d^n}{1+s_{coal}\,4\lambda_d N_d^n\Delta V_d^+}.
$$

Bulk-to-dislocation capture is active in `capture_only` and transfers a clipped fraction of bulk bubbles/gas/vacancies to the dislocation population:

$$
f_{cap}=\min\left(1,\max\left(0,s_{cap} N_d\,4\pi(R_d+R_b)^2\Delta R_d^+\right)\right).
$$

The v14-specific dislocation density law is

$$
\rho_d(F,T)=\rho_{scale}\max\left(\rho_{fab}, C_1\max(F-F_0,0)\right)f_T^{sat}(T),
$$

with the saturating Ray-Blank temperature shape normalized at 1025 K.

In [ ]:
from pathlib import Path
import csv
import importlib.util
import sys
import math

import matplotlib.pyplot as plt

SCRIPT_NAME = 'UN_M7_optuna_calibration_v14_rhoSat_qgbStrict_NdAnchors_STANDALONE.py'

# Works when the notebook is launched either from this folder or from the repo root.
search_roots = [Path.cwd(), *Path.cwd().parents]
script_path = None
for root in search_roots:
    candidates = [
        root / SCRIPT_NAME,
        root / 'UN_model' / 'optuna' / 'M7_v14' / SCRIPT_NAME,
    ]
    for candidate in candidates:
        if candidate.exists():
            script_path = candidate.resolve()
            break
    if script_path:
        break

if script_path is None:
    raise FileNotFoundError(f'Could not find {SCRIPT_NAME}')

MODEL_DIR = script_path.parent
RESULTS_DIR = MODEL_DIR / 'UN_M7_optuna_v14_rhoSat_qgbStrict_NdAnchors_results' / 'capture_only'
PLOTS_DIR = MODEL_DIR / 'notebook_plots'
PLOTS_DIR.mkdir(exist_ok=True)

spec = importlib.util.spec_from_file_location('m7_v14_standalone', script_path)
m7 = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = m7
spec.loader.exec_module(m7)

m7.set_model_family('capture_only')
print(f'Loaded solver: {script_path}')
print(f'Results dir:   {RESULTS_DIR}')

In [ ]:
FINAL_CSV = RESULTS_DIR / 'optuna_final_top_capture_only.csv'
FAST_CSV = RESULTS_DIR / 'optuna_fast_trials_capture_only.csv'

with FINAL_CSV.open(newline='', encoding='utf-8') as f:
    final_rows = list(csv.DictReader(f))

if not final_rows:
    raise RuntimeError(f'No final rows found in {FINAL_CSV}')

best_row = min(final_rows, key=lambda r: float(r['score_total']))
cand = m7.candidate_from_score_row_v10c(best_row, label_prefix='notebook_best')

DT_H = 1.0
N_MODES = 40

print('Selected candidate')
print('  label       =', best_row['label'])
print('  score_total =', float(best_row['score_total']))
print('  rho_scale   =', float(best_row.get('rho_scale', 1.0)))
print('  f_n         =', cand.f_n)
print('  K_d         =', cand.K_d)
print('  Fdot        =', cand.fission_rate)

In [ ]:
scaling_factors = {
    'rho_scale': float(best_row.get('rho_scale', 1.0)),
    'Dg_D1_scale': cand.Dg_D1_scale,
    'Dg_D3_scale': cand.Dg_D3_scale,
    'Dv_D1_scale': cand.Dv_D1_scale,
    'Dv_D2_scale': cand.Dv_D2_scale,
    'b_bulk_scale': cand.b_bulk_scale,
    'b_dislocation_scale': cand.b_dislocation_scale,
    'gb_scale': cand.gb_scale,
    'gd_bubble_scale': cand.gd_bubble_scale,
    'gd_line_scale': cand.gd_line_scale,
    'gd_line_alpha': cand.gd_line_alpha,
    'coalescence_d_scale': cand.coalescence_d_scale,
    'capture_scale': cand.capture_scale,
}

for name, value in scaling_factors.items():
    print(f'{name:24s} {value:.6g}')

print('\nEffective rho_d examples')
for bu in [1.1, 1.3, 3.2, 6.0]:
    vals = [m7.rho_ray_blank_eff_FT2(T, bu, cand) for T in [1025.0, 1600.0, 2000.0]]
    print(f'{bu:3.1f}% FIMA:', '  '.join(f'{v:.3e}' for v in vals))

In [ ]:
def run_point(T, burnup, keep_history=False, dt_h=DT_H, n_modes=N_MODES):
    return m7.run_model_point_rhoFT2(T, burnup, cand, dt_h, n_modes, keep_history=keep_history)

TEMPS = [float(T) for T in range(900, 2001, 50)]
BURNUPS = [1.1, 1.3, 3.2]

series = {
    bu: [run_point(T, bu, keep_history=False) for T in TEMPS]
    for bu in BURNUPS
}

print('Computed temperature sweeps:', {bu: len(rows) for bu, rows in series.items()})

In [ ]:
def savefig(name):
    path = PLOTS_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=180)
    print(f'saved {path}')

# Swelling vs temperature.
plt.figure(figsize=(7, 4.5))
for bu, rows in series.items():
    plt.plot(TEMPS, [r['swelling_d_percent'] for r in rows], label=f'dislocation {bu}% FIMA')
    plt.plot(TEMPS, [r['swelling_b_percent'] for r in rows], '--', label=f'bulk {bu}% FIMA')
for p in m7.EXP_SWELLING_T:
    if p['burnup'] in BURNUPS:
        plt.scatter(p['T'], p['swelling'], s=30, color='black', alpha=0.55)
plt.xlabel('Temperature [K]')
plt.ylabel('Swelling [%]')
plt.title('M7 v14 swelling replay')
plt.legend(ncol=2, fontsize=8)
savefig('swelling_vs_temperature.png')
plt.show()

In [ ]:
# Radius and number density at 1.3% FIMA.
rows = series[1.3]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(TEMPS, [r['Rd_nm'] for r in rows], label='R_d')
axes[0].plot(TEMPS, [r['Rb_nm'] for r in rows], label='R_b')
axes[0].scatter([p['T'] for p in m7.EXP_RD_T_13], [p['R_nm'] for p in m7.EXP_RD_T_13], color='black', s=35, label='exp R_d')
axes[0].set_xlabel('Temperature [K]')
axes[0].set_ylabel('Radius [nm]')
axes[0].set_title('Bubble radius, 1.3% FIMA')
axes[0].legend()

axes[1].semilogy(TEMPS, [r['Nd'] for r in rows], label='N_d')
axes[1].semilogy(TEMPS, [r['Nb'] for r in rows], label='N_b')
axes[1].scatter([p['T'] for p in m7.EXP_ND_T_13], [p['N'] for p in m7.EXP_ND_T_13], color='black', s=35, label='exp N_d')
axes[1].set_xlabel('Temperature [K]')
axes[1].set_ylabel('Number density [m$^{-3}$]')
axes[1].set_title('Bubble density, 1.3% FIMA')
axes[1].legend()
savefig('radius_and_density_1p3FIMA.png')
plt.show()

In [ ]:
# Gas partition and pressure ratios.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for bu, rows in series.items():
    axes[0].plot(TEMPS, [r['bulk_gas_percent'] for r in rows], '--', label=f'bulk {bu}%')
    axes[0].plot(TEMPS, [r['dislocation_gas_percent'] for r in rows], label=f'disl {bu}%')
    axes[0].plot(TEMPS, [r['qgb_gas_percent'] for r in rows], ':', label=f'qgb {bu}%')
axes[0].set_xlabel('Temperature [K]')
axes[0].set_ylabel('Gas inventory [% generated]')
axes[0].set_title('Gas partition')
axes[0].legend(fontsize=7, ncol=2)

for bu, rows in series.items():
    axes[1].semilogy(TEMPS, [r['p_d_over_eq'] for r in rows], label=f'p_d/p_eq {bu}%')
    axes[1].semilogy(TEMPS, [r['p_b_over_eq'] for r in rows], '--', label=f'p_b/p_eq {bu}%')
axes[1].axhspan(1/3, 3, color='0.9', zorder=0)
axes[1].set_xlabel('Temperature [K]')
axes[1].set_ylabel('Pressure ratio')
axes[1].set_title('Pressure sanity band')
axes[1].legend(fontsize=7, ncol=2)
savefig('gas_partition_and_pressure.png')
plt.show()

In [ ]:
# Burnup scan and one history trace.
burnups = [round(0.2 + 0.2*i, 2) for i in range(31)]
scan_1600 = [run_point(1600.0, bu, keep_history=False) for bu in burnups]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
axes[0].plot(burnups, [r['swelling_d_percent'] for r in scan_1600], label='dislocation')
axes[0].plot(burnups, [r['swelling_b_percent'] for r in scan_1600], label='bulk')
axes[0].plot(burnups, [r['swelling_ig_percent'] for r in scan_1600], label='total')
axes[0].set_xlabel('Burnup [% FIMA]')
axes[0].set_ylabel('Swelling [%]')
axes[0].set_title('1600 K burnup scan')
axes[0].legend()

hist = run_point(1600.0, 3.2, keep_history=True)['hist']
axes[1].plot(hist['burnup_percent_fima'], [100*x for x in hist['swelling_d']], label='dislocation')
axes[1].plot(hist['burnup_percent_fima'], [100*x for x in hist['swelling_b']], label='bulk')
axes[1].plot(hist['burnup_percent_fima'], [100*x for x in hist['swelling_ig']], label='total')
axes[1].set_xlabel('Burnup [% FIMA]')
axes[1].set_ylabel('Swelling [%]')
axes[1].set_title('History: 1600 K, 3.2% FIMA')
axes[1].legend()
savefig('burnup_scan_and_history.png')
plt.show()

In [ ]:
print(f'All notebook plots are in: {PLOTS_DIR}')